In [1]:
import numpy as np
import pandas as pd

In [5]:
df=pd.read_csv(r"C:\Users\User\Documents\Sentiment analysis\dataset\IMDB Dataset.csv")
df.head(5)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Text cleaning

In [8]:
import re
from bs4 import BeautifulSoup

def clean_review(text):
    # 1. Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # 2. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Keep only letters and basic punctuation, lowercase it
    # We keep ! and ? because they indicate strong sentiment
    text = re.sub(r"[^a-zA-Z!?,.]", " ", text)
    text = text.lower().strip()

    # 4. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    return text


df['review'] = df['review'].apply(clean_review)

In [9]:
df["review"]

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there s a family where a little boy ...
4        petter mattei s love in the time of money is a...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i m going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: object

Vectorization

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000,ngram_range=(1,2))
X=vectorizer.fit_transform(df['review'])


Label Encoding


In [45]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(df['sentiment'])


Model training

In [46]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)


In [47]:

from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:,1]
y_pred_new=(y_probs>0.5).astype(int)

In [48]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
cm = confusion_matrix(y_test, y_pred_new)




In [50]:
print("Confusion Matrix")
print(cm)
print("Classification Report")
print(classification_report(y_test, y_pred_new))
print("Accuracy Score")
print(accuracy_score(y_test, y_pred_new))

Confusion Matrix
[[4395  566]
 [ 460 4579]]
Classification Report
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000

Accuracy Score
0.8974
